# 02. Modern LangGraph Text-to-SQL

This notebook demonstrates the v0.3 agent by importing the production package. No
graph, prompt, or validation rule is redefined here; everything shown is the code
that runs in `src/enterprise_agents_on_foundry/agents/`.

The legacy agent could only be observed by calling a live model against a live
database. Most of what follows runs offline, because the graph takes its model and
its database as callables. The live section is clearly marked and skips itself when
the environment is unavailable.

The order of the sections matches the order in which the design decisions were
made: state first, then topology, then behaviour, then what was left out.

In [1]:
from __future__ import annotations

import json
import time
from typing import Any

from enterprise_agents_on_foundry.agents.graph import (
    compile_graph,
    route_after_execution,
    route_after_repair,
    route_after_validation,
)
from enterprise_agents_on_foundry.agents.nodes import (
    QUERY_LABEL,
    RECURSION_LIMIT,
    AgentDependencies,
    SqlText,
    answer_question,
    production_nodes,
)
from enterprise_agents_on_foundry.agents.prompts import SQL_SYSTEM_PROMPT
from enterprise_agents_on_foundry.agents.state import (
    DEFAULT_MAX_ROWS,
    MAX_REPAIR_ATTEMPTS,
    AgentInput,
    AgentState,
    NodeName,
    SqlDraft,
    initial_state,
)
from enterprise_agents_on_foundry.config.settings import repository_root
from enterprise_agents_on_foundry.database.models import QueryRequest, QueryResult
from enterprise_agents_on_foundry.errors import ModelOutputError
from enterprise_agents_on_foundry.observability.measurements import MeasurementSet

measurements = MeasurementSet(release="v0.3")
print(f"repair budget:   {MAX_REPAIR_ATTEMPTS}")
print(f"default row cap: {DEFAULT_MAX_ROWS}")
print(f"recursion limit: {RECURSION_LIMIT}")

repair budget:   1
default row cap: 200
recursion limit: 9


## 1. Typed graph state

The legacy agent kept everything in a message transcript and then deleted messages
to stop the transcript growing, so its real state was implicit and no routing rule
could be tested without a model.

Three shapes are separate here. `AgentInput` is what a caller may supply, and it is
validated. `AgentState` is internal working state where every key is always present,
so absence is `None` rather than a missing key. `AgentOutput` is what a caller
receives, built by exactly one node per outcome.

In [2]:
request = AgentInput(question="Which product categories exist?", max_rows=25)
start = initial_state(request)

print(f"{'key':<20}{'declared type':<28}initial value")
print("-" * 74)
for key, declared in AgentState.__annotations__.items():
    rendered = declared if isinstance(declared, str) else getattr(declared, "__name__", str(declared))
    print(f"{key:<20}{rendered:<28}{start[key]!r}")

print(f"\nevery key present: {set(AgentState.__annotations__) == set(start)}")
print(f"nothing optional:  {AgentState.__optional_keys__ == frozenset()}")

key                 declared type               initial value
--------------------------------------------------------------------------
question            ForwardRef('str', module='enterprise_agents_on_foundry.agents.state')'Which product categories exist?'
max_rows            ForwardRef('int', module='enterprise_agents_on_foundry.agents.state')25
schema_context      ForwardRef('str', module='enterprise_agents_on_foundry.agents.state')''
sql                 ForwardRef('str | None', module='enterprise_agents_on_foundry.agents.state')None
rationale           ForwardRef('str | None', module='enterprise_agents_on_foundry.agents.state')None
validation_error    ForwardRef('str | None', module='enterprise_agents_on_foundry.agents.state')None
execution_error     ForwardRef('str | None', module='enterprise_agents_on_foundry.agents.state')None
repair_attempts     ForwardRef('int', module='enterprise_agents_on_foundry.agents.state')0
result              ForwardRef('QueryResult | None', module='

## 2. Nodes and edges

Seven nodes, named by an enumeration so a typo in an edge definition is an error at
import time rather than a routing bug at run time.

Routing is three pure functions over state. They take no model and no database, so
the whole control flow is exercisable with plain dictionaries. The table below does
exactly that: it states a situation and prints where the graph would go.

In [3]:
def situation(**overrides: Any) -> AgentState:
    """Build a state that describes one situation."""
    state = dict(start)
    state.update(overrides)
    return state  # type: ignore[return-value]


cases = [
    ("validation passed", route_after_validation, situation(validation_error=None)),
    ("validation failed, budget left", route_after_validation, situation(validation_error="forbidden keyword")),
    ("validation failed, budget spent", route_after_validation, situation(validation_error="x", repair_attempts=1)),
    ("execution succeeded", route_after_execution, situation(execution_error=None)),
    ("execution failed, budget left", route_after_execution, situation(execution_error="invalid column")),
    ("execution failed, budget spent", route_after_execution, situation(execution_error="x", repair_attempts=1)),
    ("repair produced a statement", route_after_repair, situation(sql="SELECT 1")),
    ("repair produced nothing", route_after_repair, situation(sql=None)),
]

print(f"{'situation':<34}{'goes to':<18}decided by")
print("-" * 74)
for label, rule, state in cases:
    print(f"{label:<34}{rule(state).value:<18}{rule.__name__}")

print(f"\nnode names: {[name.value for name in NodeName]}")

situation                         goes to           decided by
--------------------------------------------------------------------------
validation passed                 execute_sql       route_after_validation
validation failed, budget left    repair_sql        route_after_validation
validation failed, budget spent   fail              route_after_validation
execution succeeded               compose_answer    route_after_execution
execution failed, budget left     repair_sql        route_after_execution
execution failed, budget spent    fail              route_after_execution
repair produced a statement       validate_sql      route_after_repair
repair produced nothing           fail              route_after_repair

node names: ['load_schema', 'generate_sql', 'validate_sql', 'execute_sql', 'repair_sql', 'compose_answer', 'fail']


## 3. Graph visualization

The topology is fixed when the graph compiles, so it can be drawn without running
anything. A repair is a single bounded detour, not a loop the model can re-enter at
will, and both terminal nodes reach `END`.

```mermaid
stateDiagram-v2
    [*] --> load_schema
    load_schema --> generate_sql
    generate_sql --> validate_sql
    validate_sql --> execute_sql: valid
    validate_sql --> repair_sql: invalid, budget left
    validate_sql --> fail: invalid, budget spent
    execute_sql --> compose_answer: rows returned
    execute_sql --> repair_sql: error, budget left
    execute_sql --> fail: error, budget spent
    repair_sql --> validate_sql: statement produced
    repair_sql --> fail: nothing produced
    compose_answer --> [*]
    fail --> [*]
```

The next cell defines the scripted model and database used by every offline
demonstration, then prints the diagram LangGraph generates from the compiled graph.

In [4]:
class ScriptedModel:
    """Replays scripted drafts and records how many times it was called."""

    def __init__(self, drafts: list[SqlText | ModelOutputError], answer: str = "There are four categories.") -> None:
        self._drafts = list(drafts)
        self._answer = answer
        self.calls = 0

    def draft(self, system: str, user: str) -> SqlText:
        self.calls += 1
        response = self._drafts.pop(0)
        if isinstance(response, ModelOutputError):
            raise response
        return response

    def write(self, system: str, user: str) -> str:
        self.calls += 1
        return self._answer


class ScriptedDatabase:
    """Replays scripted outcomes and records every request that reached it."""

    def __init__(self, outcomes: list[QueryResult | Exception] | None = None) -> None:
        self._outcomes = list(outcomes) if outcomes else []
        self.requests: list[QueryRequest] = []

    def run(self, request: QueryRequest) -> QueryResult:
        self.requests.append(request)
        outcome = self._outcomes.pop(0) if self._outcomes else categories()
        if isinstance(outcome, Exception):
            raise outcome
        return outcome


def categories() -> QueryResult:
    """A small result standing in for four rows from SalesLT.ProductCategory."""
    return QueryResult(
        columns=("Name",),
        rows=(("Bikes",), ("Components",), ("Clothing",), ("Accessories",)),
        truncated=False,
        elapsed_ms=4.0,
        label=QUERY_LABEL,
    )


def scripted(model: ScriptedModel, database: ScriptedDatabase) -> AgentDependencies:
    """Wire a scripted model and database into the production node contracts."""
    return AgentDependencies(
        load_schema=lambda: "SalesLT.ProductCategory\n  Name nvarchar not null",
        draft_sql=model.draft,
        write_answer=model.write,
        run_query=database.run,
    )


topology = compile_graph(production_nodes(scripted(ScriptedModel([]), ScriptedDatabase()))).get_graph()
node_count = len([name for name in topology.nodes if name not in {"__start__", "__end__"}])
edge_count = len(topology.edges)

print(topology.draw_mermaid())
print(f"nodes: {node_count}   edges: {edge_count}")

measurements.add("graph nodes", node_count, category="graph", baseline=2, note="v0.1 had agent and tools")
measurements.add("graph edges", edge_count, category="graph", baseline=3, note="v0.1 wired start, agent, tools")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	load_schema(load_schema)
	generate_sql(generate_sql)
	validate_sql(validate_sql)
	execute_sql(execute_sql)
	repair_sql(repair_sql)
	compose_answer(compose_answer)
	fail(fail)
	__end__([<p>__end__</p>]):::last
	__start__ --> load_schema;
	execute_sql -.-> compose_answer;
	execute_sql -.-> fail;
	execute_sql -.-> repair_sql;
	generate_sql --> validate_sql;
	load_schema --> generate_sql;
	repair_sql -.-> fail;
	repair_sql -.-> validate_sql;
	validate_sql -.-> execute_sql;
	validate_sql -.-> fail;
	validate_sql -.-> repair_sql;
	compose_answer --> __end__;
	fail --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

nodes: 7   edges: 13


## 4. Successful Text-to-SQL execution

The happy path costs two model calls: one to draft the statement and one to describe
the rows. The legacy agent spent at least four, because the model had to call
`list_tables` and `get_schema` before it could write anything, and nothing stopped it
calling them again.

The row cap is applied by the request, not by asking the model to add `TOP (n)`.

In [5]:
GOOD_SQL = "SELECT TOP (25) Name FROM SalesLT.ProductCategory"

model = ScriptedModel([SqlText(sql=GOOD_SQL, rationale="lists the categories")])
database = ScriptedDatabase()

started = time.perf_counter()
output = answer_question(scripted(model, database), request)
happy_ms = round((time.perf_counter() - started) * 1000, 1)

print(f"status:        {output.status.value}")
print(f"answer:        {output.answer}")
print(f"sql:           {output.sql}")
print(f"rows:          {output.row_count}")
print(f"repairs:       {output.repair_attempts}")
print(f"model calls:   {model.calls}")
print(f"row cap sent:  {database.requests[0].max_rows}")
print(f"query label:   {database.requests[0].label}")

measurements.add(
    "model calls, happy path",
    model.calls,
    category="agent",
    baseline=4,
    note="v0.1 called list_tables and get_schema before drafting",
)

status:        succeeded


answer:        There are four categories.
sql:           SELECT TOP (25) Name FROM SalesLT.ProductCategory
rows:          4
repairs:       0
model calls:   2
row cap sent:  25
query label:   agent


## 5. Invalid-SQL repair

A statement that fails validation, or fails to run, is corrected exactly once. The
repair prompt receives the failed statement and the verbatim error, because a
validator message names the keyword it refused and a database message names the
object it could not resolve.

A repaired statement goes back through validation, never straight to execution. It
is model output, and therefore exactly as untrusted as the statement it replaced.

In [6]:
BROKEN_SQL = "SELECT Name FROM SalesLT.ProductCategory; SELECT 1"

model = ScriptedModel([SqlText(sql=BROKEN_SQL, rationale=""), SqlText(sql=GOOD_SQL, rationale="corrected")])
database = ScriptedDatabase()

started = time.perf_counter()
repaired = answer_question(scripted(model, database), request)
repair_ms = round((time.perf_counter() - started) * 1000, 1)

print(f"status:       {repaired.status.value}")
print(f"sql:          {repaired.sql}")
print(f"repairs:      {repaired.repair_attempts}")
print(f"model calls:  {model.calls}")
print(f"executed:     {len(database.requests)}, and only after the correction was revalidated")

measurements.add(
    "model calls, repair path",
    model.calls,
    category="agent",
    note="v0.1 had no attempt counter, so this path was unbounded",
)

status:       succeeded
sql:          SELECT TOP (25) Name FROM SalesLT.ProductCategory
repairs:      1
model calls:  3
executed:     1, and only after the correction was revalidated


## 6. Unsafe-write rejection

The prompt asks for a read-only statement. Nothing depends on the model agreeing.

Validation is deterministic and runs before execution, and the database boundary
validates again before touching the driver. A mutation therefore cannot reach the
database even if the graph were rewired incorrectly.

In [7]:
UNSAFE_SQL = "DELETE FROM SalesLT.Product WHERE SellEndDate IS NOT NULL"

model = ScriptedModel([SqlText(sql=UNSAFE_SQL, rationale=""), SqlText(sql=UNSAFE_SQL, rationale="")])
database = ScriptedDatabase()

blocked = answer_question(scripted(model, database), AgentInput(question="Delete discontinued products."))

print(f"status:            {blocked.status.value}")
print(f"failure stage:     {blocked.failure_stage.value if blocked.failure_stage else '-'}")
print(f"reason:            {blocked.failure_reason}")
print(f"reached database:  {len(database.requests)}")

measurements.add(
    "unsafe statements blocked before execution",
    model.calls,
    category="safety",
    baseline=0,
    note="v0.1 executed what the model produced, after stripping code fences",
)

status:            failed
failure stage:     validation
reason:            Query must begin with SELECT or WITH; found 'DELETE'.
reached database:  0


## 7. Retry exhaustion

The cell above also showed exhaustion: two rejected statements and no third attempt.

The counter is incremented before the model is called, so a model that raises still
spends its budget. The maximum number of statements the model may write is two, and
that is fixed when the graph compiles rather than enforced by a timeout. The result
is a structured failure, not an exception and not an invented answer.

In [8]:
print(f"drafts requested:  {model.calls}, and the cap is {MAX_REPAIR_ATTEMPTS + 1}")
print(f"repairs used:      {blocked.repair_attempts}")
print(f"answer invented:   {blocked.answer is not None}")

empty_model = ScriptedModel([ModelOutputError("the model returned no statement")])
compiled = compile_graph(production_nodes(scripted(empty_model, ScriptedDatabase())))

visited: list[str] = []
for update in compiled.stream(initial_state(request), stream_mode="updates"):
    visited.extend(update)

print(f"\npath when the first draft cannot be produced at all:\n  {' -> '.join(visited)}")
print(f"drafts requested:  {empty_model.calls}, because there is nothing to repair")

measurements.add("maximum statements the model may write", MAX_REPAIR_ATTEMPTS + 1, category="agent")

drafts requested:  2, and the cap is 2
repairs used:      1
answer invented:   False

path when the first draft cannot be produced at all:
  load_schema -> generate_sql -> validate_sql -> fail
drafts requested:  1, because there is nothing to repair


## 8. Structured output

The model returns a schema-constrained object, not prose that has to be parsed.

`SqlDraft` forbids extra fields, so a response carrying a tool call is rejected
rather than partly honoured, and the model cannot smuggle in an action. A fenced
code block is rejected too, rather than stripped: under constrained decoding a fence
means something upstream is wrong, and the legacy habit of quietly removing it hid
exactly that.

In [9]:
print(json.dumps(SqlDraft.model_json_schema(), indent=2))

payloads = [
    ("valid draft", {"sql": GOOD_SQL, "rationale": "lists the categories"}),
    ("blank statement", {"sql": "   "}),
    ("smuggled tool call", {"sql": GOOD_SQL, "tool_calls": [{"name": "run"}]}),
]

print()
for description, payload in payloads:
    try:
        SqlDraft(**payload)
    except ValueError as error:
        lines = str(error).splitlines()
        print(f"{description:<20} rejected: {lines[1].strip() if len(lines) > 1 else lines[0]}")
    else:
        print(f"{description:<20} accepted")

print(f"\nsystem prompt sent for drafting:\n\n{SQL_SYSTEM_PROMPT}")

{
  "additionalProperties": false,
  "description": "The only thing the model is allowed to produce for a query.\n\nThe model returns a statement and a reason. It does not return a decision, a\ntool call, or an action. Execution is the graph's job, not the model's.",
  "properties": {
    "sql": {
      "description": "A single read-only T-SQL SELECT statement.",
      "minLength": 1,
      "title": "Sql",
      "type": "string"
    },
    "rationale": {
      "default": "",
      "description": "One line explaining the query. Never executed.",
      "title": "Rationale",
      "type": "string"
    }
  },
  "required": [
    "sql"
  ],
  "title": "SqlDraft",
  "type": "object"
}

valid draft          accepted
blank statement      rejected: sql
smuggled tool call   rejected: tool_calls

system prompt sent for drafting:

You write T-SQL for Microsoft Azure SQL Database.

Rules:
- Return exactly one SELECT statement. No batches, no semicolon-separated statements.
- Use TOP (n) to limit ro

## Live run against Azure

Everything above ran offline. This cell uses the deployed model and the provisioned
AdventureWorksLT database, and reports why it skipped when either is unavailable.

It captures the only measurement that needs Azure: end-to-end latency on the happy
path. The offline cells above measure graph overhead, which is a different number
and is labelled as such.

In [10]:
live_happy_ms: float | None = None

try:
    from enterprise_agents_on_foundry.agents.nodes import production_dependencies
    from enterprise_agents_on_foundry.config.settings import load_settings
    from enterprise_agents_on_foundry.database.connection import connect

    settings = load_settings()
    with connect(settings) as live_client:
        live_deps = production_dependencies(settings, live_client)
        started = time.perf_counter()
        live = answer_question(live_deps, AgentInput(question="Which product categories exist?", max_rows=25))
        live_happy_ms = round((time.perf_counter() - started) * 1000, 1)

    print(f"status:   {live.status.value}")
    print(f"answer:   {live.answer}")
    print(f"sql:      {live.sql}")
    print(f"rows:     {live.row_count}")
    print(f"latency:  {live_happy_ms} ms")
except Exception as error:
    print(f"skipped: {type(error).__name__}: {error}")

skipped: DatabaseConnectionError: The 'ODBC Driver 18 for SQL Server' ODBC driver is not installed.
It is a system package, so 'uv sync' cannot provide it.

Windows: winget install --id Microsoft.msodbcsql18
macOS:   brew install msodbcsql18
Linux:   https://learn.microsoft.com/sql/connect/odbc/linux-mac/installing-the-microsoft-odbc-driver-for-sql-server

Installing it requires administrator rights. Open a new terminal afterwards so the driver registration is picked up.

Drivers currently visible: SQL Server, Microsoft Access Driver (*.mdb, *.accdb), Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb), Microsoft Access Text Driver (*.txt, *.csv), Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)


## Measurements

Test counts and durations are recorded from the commands listed in the release
notes rather than by running pytest inside the notebook, so the notebook does not
re-run the suite it belongs to.

In [11]:
measurements.add("happy path latency", live_happy_ms, unit="ms", category="agent", note="end to end against Azure")
measurements.add("happy path graph overhead", happy_ms, unit="ms", category="agent", note="scripted model, graph only")
measurements.add("repair path graph overhead", repair_ms, unit="ms", category="agent", note="one correction included")
measurements.add("offline tests", 238, category="tests", baseline=213, note="uv run pytest -m 'not azure'")
measurements.add("offline test duration", 24.0, unit="s", category="tests", baseline=11.4)
measurements.add("integration tests", 14, category="tests", baseline=10, note="uv run pytest -m azure")
measurements.add("integration test duration", 8.9, unit="s", category="tests", note="13 skipped, no ODBC driver")

print(measurements.format_table())

written = measurements.write_json(repository_root() / "docs" / "releases" / "v0.3-measurements.json")
print(f"\nwritten to {written.relative_to(repository_root())}")

category    measure                               before     after  unit
------------------------------------------------------------------------
graph       graph nodes                                2         7  count
graph       graph edges                                3        13  count
agent       model calls, happy path                    4         2  count
agent       model calls, repair path                   -         3  count
safety      unsafe statements blocked before execution         0         2  count
agent       maximum statements the model may write         -         2  count
agent       happy path latency                         -         -  ms
agent       happy path graph overhead                  -      60.1  ms
agent       repair path graph overhead                 -      34.8  ms
tests       offline tests                            213       238  count
tests       offline test duration                   11.4      24.0  s
tests       integration tests            

## 9. Limitations and deferred capabilities

What this agent cannot do, and why that is deliberate.

* One question, one answer. There is no checkpointer and no thread, so a follow-up
  question does not know what the previous one asked. Persistence is v0.4.
* The whole schema is rendered into the prompt. That works because AdventureWorksLT
  is small. Retrieval over a schema only earns its complexity on a database too
  large to render.
* One repair, then failure. A model that writes broken SQL twice is unlikely to be
  talked round by a third prompt, and an unbounded loop is what made the legacy
  agent unpredictable.
* The validator matches keywords and is conservative, so a legitimate query that
  contains a forbidden word inside a string literal is refused. Refusing a valid
  query is a worse experience than allowing one, and a much better failure than the
  reverse.
* Answer quality is not measured. There is no evaluation runner yet, so nothing here
  claims the answers are correct, only that they are grounded in rows that were
  actually retrieved.
* Two properties of the deployment are pinned as constants in `agents/model.py`
  rather than discovered at run time: the structured-output method and whether the
  deployment accepts an explicit temperature. Changing a model version means
  rechecking both.
* No streaming, no memory, no hosted deployment, no FastAPI, no user interface, no
  MCP, no Foundry IQ, and no new Azure resources. Each has its own release.

Next: `docs/architecture/v0.3-modern-langgraph-agent.md` for the design and the
alternatives that were rejected, and `docs/releases/v0.3.md` for the measurements
and the verification commands.